# Issue 76 – Dialogqualität Schritt für Schritt prüfen

Dieses Notebook prüft fünf festgelegte Gesprächssituationen. Es verändert keine Falldaten und keinen Produktcode. Die Ergebnisse werden unter `output/evals/issue76_results.json` gespeichert.

**Reihenfolge:** Führe die Zellen von oben nach unten einzeln mit **Shift + Enter** aus. Wenn eine Zelle fertig ist, erscheint links eine Zahl statt `[*]`.


## Schritt 1 – Testumgebung vorbereiten

**Wann ausführen?** Einmal nach dem Öffnen des Notebooks oder nach einem Neustart des Kernels.

Die Zelle findet das Projekt, lädt den Testläufer und liest die fünf Testfälle ein. Erwartetes Ergebnis: `5 Testfälle geladen.`


In [ ]:
from pathlib import Path
import sys
from IPython.display import HTML, display

# Projektordner finden – unabhängig davon, ob VS Code im Projekt oder im Notebook-Ordner gestartet wurde.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'sob_generator').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evals.issue76_runner import load_cases, run_suite, write_results

cases = load_cases()
print(f'{len(cases)} Testfälle geladen.')


## Schritt 2 – Testart auswählen und Test ausführen

**Für einen schnellen technischen Probelauf:** `ECHTES_MODELL = False`. Das Ergebnis kommt fast sofort, bewertet aber nicht die Qualität des Chatbots.

**Für die fachliche Prüfung der App:** `ECHTES_MODELL = True`. Verwende diese Einstellung nach einer Änderung am Chat. Das lokale 7B-Modell wird geladen; der erste Lauf kann etwas dauern.

Führe diese Zelle nach jeder Änderung erneut aus. Erwartetes Ergebnis: Der Speicherort der Ergebnisdatei wird angezeigt.


In [ ]:
ECHTES_MODELL = True  # True = echter Chatbot; False = schneller technischer Probelauf

if ECHTES_MODELL:
    from sob_generator.llm.generator import LLMDraftGenerator
    modellpfad = REPO_ROOT / 'models/Qwen2.5-7B-Instruct-Q4_K_M.gguf'
    if not modellpfad.exists():
        raise FileNotFoundError(f'Modell nicht gefunden: {modellpfad}')
    generator = LLMDraftGenerator(str(modellpfad), verbose=False)
    print('Fachliche Prüfung mit dem echten 7B-Modell läuft …')
else:
    from sob_generator.stub_generator import StubDraftGenerator
    generator = StubDraftGenerator()
    print('Schneller technischer Probelauf läuft …')

# Echte Modellantworten für die Abnahme immer neu erzeugen.
results = await run_suite(cases, generator, use_cache=not ECHTES_MODELL)
results_path = write_results(results)
print(f'Ergebnisse gespeichert: {results_path}')


## Schritt 3 – Ergebnisübersicht anzeigen

**Wann ausführen?** Direkt nach Schritt 2.

- `✅ Bestanden`: Die festgelegten fachlichen Erwartungen wurden erfüllt.
- `❌ Prüfen`: Mindestens eine Erwartung wurde verletzt. Der Befund nennt den konkreten Grund.
- Eine Laufzeit von mehreren Sekunden zeigt, dass das echte Modell verwendet wurde. Bei einem zwischengespeicherten Ergebnis kann der Lauf schneller sein.


In [ ]:
rows = []
for result in results:
    status = '✅ Bestanden' if result.passed else '❌ Prüfen'
    problems = '<br>'.join(result.violations) or '–'
    rows.append(
        f'<tr><td>{result.case_id}</td><td>{status}</td>'
        f'<td>{result.duration_seconds:.2f} s</td><td>{problems}</td></tr>'
    )

display(HTML(
    '<table><thead><tr><th>Fall</th><th>Status</th><th>Zeit</th><th>Befund</th></tr></thead>'
    '<tbody>' + ''.join(rows) + '</tbody></table>'
))


## Schritt 4 – Fehlgeschlagene Gespräche lesen

**Wann ausführen?** Wenn Schritt 3 mindestens einen Fall mit `❌ Prüfen` zeigt.

Die Zelle zeigt nur die Antworten der fehlgeschlagenen Fälle. Prüfe dabei:

1. Bleibt die Antwort bei den tatsächlich genannten Informationen?
2. Ist die Antwort fachlich sinnvoll?
3. Versteht eine Fachkraft sofort, was sie als Nächstes tun soll?

Die Datei mit allen strukturierten Ergebnissen findest du unter `output/evals/issue76_results.json`.


In [ ]:
failed_results = [result for result in results if not result.passed]

if not failed_results:
    print('Alle Testfälle wurden bestanden.')
else:
    for result in failed_results:
        print(f'\n### {result.case_id}')
        print('Befund:')
        for violation in result.violations:
            print(f'- {violation}')
        print('Antworten des Chatbots:')
        for number, reply in enumerate(result.replies, start=1):
            print(f'{number}. {reply}')


# Interaktiver Chat – eigene Eingaben prüfen

Die Tabelle oben ist der automatische Sicherheitstest. In diesem Bereich führst du selbst ein fortlaufendes Gespräch mit dem echten Chatbot.

## Schritt 5 – Neues Testgespräch starten

**Wann ausführen?** Einmal vor deinem manuellen Test oder wenn du das Gespräch vollständig zurücksetzen möchtest.

Voraussetzung: In Schritt 2 muss `ECHTES_MODELL = True` eingestellt und ausgeführt worden sein. Das Testgespräch verwendet ausschließlich synthetische Angaben zu „Herrn Mustermann“.


In [ ]:
from sob_generator.chat_engine import ChatDialogueEngine
from sob_generator.chat_session import ChatSession, DialoguePhase, SectionChatState
from sob_generator.chat_store import ChatSessionStore
from sob_generator.models import ClientData

chat_session = ChatSession(
    client=ClientData(name='Mustermann', vorname='Max', anrede='Herr'),
    phase=DialoguePhase.FIELD_REFLECT,
    active_section_id='1.1',
    sections={
        '1.1': SectionChatState(
            section_id='1.1',
            title='Wohnsituation',
        )
    },
)
chat_store = ChatSessionStore()
chat_store.store(chat_session)
chat_engine = ChatDialogueEngine(generator, session_store=chat_store)

print('Neues Testgespräch gestartet: Lebensfeld 1.1 – Wohnsituation')


## Schritt 6 – Eine Nachricht an den Chatbot senden

1. Ersetze nur den Text zwischen den Anführungszeichen bei `NACHRICHT`.
2. Führe die Zelle mit **Shift + Enter** aus.
3. Lies die Antwort unter der Zelle.
4. Ändere `NACHRICHT` und führe **dieselbe Zelle erneut** aus. Der Gesprächsverlauf bleibt dabei erhalten.

Beispiele: `Beim letzten Besuch öffnete Herr Mustermann den Brief selbst.` oder danach `Korrektur: Beim letzten Besuch öffnete die Assistenz den Brief.`


In [ ]:
NACHRICHT = 'Beim letzten Besuch öffnete Herr Mustermann den Brief selbst.'

if not NACHRICHT.strip():
    raise ValueError('Bitte eine Nachricht zwischen den Anführungszeichen eintragen.')

vorherige_anzahl = len(chat_session.messages)
chat_session = await chat_engine.process_message(chat_session.session_id, NACHRICHT)
neue_nachrichten = chat_session.messages[vorherige_anzahl:]

print(f'Fachkraft: {NACHRICHT}')
for nachricht in neue_nachrichten:
    if nachricht.role.value == 'assistant':
        print(f'\nChatbot: {nachricht.content}')


## Schritt 7 – Internen Gesprächsstand kontrollieren

**Wann ausführen?** Nach einer Antwort, wenn du prüfen möchtest, was der Chatbot tatsächlich gespeichert hat. Das ist besonders bei Korrekturen wichtig.


In [ ]:
stand = chat_session.sections[chat_session.active_section_id]
print('Gespeicherte Beobachtungen:')
if stand.user_notes:
    for notiz in stand.user_notes:
        print(f'- {notiz}')
else:
    print('– keine –')

print('\nAktueller Zielvorschlag:')
print(stand.current_goals or '– keiner –')
print(f'\nOffene Unsicherheiten:')
if stand.uncertainties:
    for unsicherheit in stand.uncertainties:
        print(f'- {unsicherheit}')
else:
    print('– keine –')
